<a href="https://colab.research.google.com/github/Kareem2266/Dungeon-Game/blob/main/p1-memory-breakdown/p1_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Nov 26 00:19:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
!git clone https://{token}@github.com/Kareem2266/ml-memory-notes.git
%cd /content/ml-memory-notes/p1-memory-breakdown
!git config --global user.name "Kareem Maamoun"
!git config --global user.email "kareemmaamoun2@gmail.com"

Cloning into 'ml-memory-notes'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 18 (delta 3), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 45.64 KiB | 834.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/ml-memory-notes/p1-memory-breakdown


In [15]:
%cd /content/ml-memory-notes/p1-memory-breakdown

path = "p1_memory_baseline.py"
src = open(path).read()
marker = "# --- Sweep context length x batch size"

warmup = '''# --- Warm-up: pay one-time costs (e.g. cuBLAS workspace) before measuring ---
before_warmup = torch.cuda.memory_allocated()
with torch.inference_mode():
    warm_ids = torch.randint(0, cfg.vocab_size, (1, 128), device=dev)
    warm_out = model.model(input_ids=warm_ids, use_cache=True)
del warm_out, warm_ids
torch.cuda.synchronize()
torch.cuda.empty_cache()
print(f"warm-up left {(torch.cuda.memory_allocated() - before_warmup) / MiB:.2f} MiB "
      f"allocated permanently")

'''

assert marker in src, "Couldn't find the sweep section"
assert "Warm-up" not in src, "Warm-up already added"
src = src.replace(marker, warmup + marker, 1)
open(path, "w").write(src)
print("Warm-up added")

/content/ml-memory-notes/p1-memory-breakdown
Warm-up added


In [16]:
!ls /content/ml-memory-notes/p1-memory-breakdown
%cd /content/ml-memory-notes/p1-memory-breakdown
!python p1_memory_baseline.py

memory_breakdown.png  p1_memory_baseline.py  results.csv  writeup.md
/content/ml-memory-notes/p1-memory-breakdown
GPU: Tesla T4 | torch 2.14.1+cu130 | transformers 5.22.1 | python 3.13.16
layers=24 kv_heads=2 head_dim=64 -> 12.0 KiB of KV cache per token
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 367.70it/s]
params=494M | weights expected 942 MiB, measured 950 MiB
warm-up left 8.12 MiB allocated permanently
batch= 1 seq=  512: KV expected     6.0 MiB, measured     6.0 MiB (+0.0%), peak total   974.0 MiB
batch= 1 seq= 1024: KV expected    12.0 MiB, measured    12.0 MiB (+0.0%), peak total   998.9 MiB
batch= 1 seq= 2048: KV expected    24.0 MiB, measured    24.0 MiB (+0.0%), peak total  1048.7 MiB
batch= 1 seq= 4096: KV expected    48.0 MiB, measured    48.0 MiB (+0.0%), peak total  1141.2 MiB
batch= 1 seq= 8192: KV expected    96.0 MiB, measured    96.0 MiB (+0.0%), peak total  1332.3 MiB
batch= 4 seq=  512: KV expected  

In [18]:
%cd /content/ml-memory-notes/p1-memory-breakdown
!mkdir -p results
!mv results.csv results/run2_with_warmup.csv
!mv memory_breakdown.png results/run2_with_warmup.png
!git add p1_memory_baseline.py results/
!git commit -m "Project 1: add warm-up pass; run 2 confirms cuBLAS workspace (8.12 MiB)"
!git push

/content/ml-memory-notes/p1-memory-breakdown
[main 7cfbbf0] Project 1: add warm-up pass; run 2 confirms cuBLAS workspace (8.12 MiB)
 3 files changed, 27 insertions(+)
 create mode 100644 p1-memory-breakdown/results/run2_with_warmup.csv
 create mode 100644 p1-memory-breakdown/results/run2_with_warmup.png
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 42.64 KiB | 14.21 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Kareem2266/ml-memory-notes.git
   5756987..7cfbbf0  main -> main


In [4]:
%cd /content/ml-memory-notes/p1-memory-breakdown
import torch
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B"
MiB = 1024 ** 2

# Start recording every GPU allocation
torch.cuda.memory._record_memory_history(max_entries=100_000)

before = torch.cuda.memory_allocated()
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
torch.cuda.synchronize()
allocated = torch.cuda.memory_allocated() - before

param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
print(f"allocated by load : {allocated / MiB:8.2f} MiB")
print(f"parameters        : {param_bytes / MiB:8.2f} MiB")
print(f"buffers           : {buffer_bytes / MiB:8.2f} MiB")
print(f"unexplained       : {(allocated - param_bytes - buffer_bytes) / MiB:8.2f} MiB")

# Inspect every live block of GPU memory
snap = torch.cuda.memory._snapshot()
ptrs = sorted(t.data_ptr() for t in list(model.parameters()) + list(model.buffers()))

def holds_model_tensor(blk):
    start, end = blk["address"], blk["address"] + blk["size"]
    return any(start <= p < end for p in ptrs)

rounding, others = 0, []
for seg in snap["segments"]:
    for blk in seg["blocks"]:
        if blk["state"] != "active_allocated":
            continue
        rounding += blk["size"] - blk.get("requested_size", blk["size"])
        if not holds_model_tensor(blk):
            others.append(blk)

print(f"allocator rounding: {rounding / MiB:8.2f} MiB")
print(f"blocks not holding model tensors: {len(others)}")
for blk in sorted(others, key=lambda b: -b["size"])[:10]:
    where = next((f"{f['filename'].split('/')[-1]}:{f['line']} in {f['name']}"
                  for f in blk.get("frames", []) if "torch/" not in f["filename"]),
                 "unknown")
    print(f"  {blk['size'] / MiB:7.2f} MiB  created at {where}")

# Save the full recording for the visual timeline (next piece)
torch.cuda.memory._dump_snapshot("load_snapshot.pickle")
torch.cuda.memory._record_memory_history(enabled=None)

/content/ml-memory-notes/p1-memory-breakdown


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

allocated by load :   950.17 MiB
parameters        :   942.29 MiB
buffers           :     0.00 MiB
unexplained       :     7.87 MiB
allocator rounding:     7.87 MiB
blocks not holding model tensors: 0


In [5]:
slack = []
for seg in snap["segments"]:
    for blk in seg["blocks"]:
        if blk["state"] == "active_allocated" and "requested_size" in blk:
            slack.append((blk["size"] - blk["requested_size"], blk["requested_size"]))
slack.sort(reverse=True)

print("blocks with any slack:", sum(1 for s, _ in slack if s > 0), "of", len(slack))
for s, req in slack[:10]:
    print(f"  requested {req / MiB:8.3f} MiB, given {(req + s) / MiB:8.3f} MiB, slack {s / MiB:6.3f} MiB")

blocks with any slack: 148 of 292
  requested  259.656 MiB, given  260.000 MiB, slack  0.344 MiB
  requested    8.312 MiB, given    8.625 MiB, slack  0.312 MiB
  requested    8.312 MiB, given    8.625 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB
  requested    1.531 MiB, given    1.844 MiB, slack  0.312 MiB


In [6]:
import torch

cfg = model.config
MiB = 1024 ** 2
batch, seq = 16, 8192

# Warm-up so one-time costs don't pollute the measurement
with torch.inference_mode():
    model.model(input_ids=torch.randint(0, cfg.vocab_size, (1, 128), device="cuda"))
torch.cuda.synchronize()
torch.cuda.empty_cache()

def prefill(chunk):
    """Read the whole prompt, `chunk` tokens per conversation at a time."""
    torch.cuda.empty_cache()
    ids = torch.randint(0, cfg.vocab_size, (batch, seq), device="cuda")
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    cache = None
    with torch.inference_mode():
        for start in range(0, seq, chunk):
            out = model.model(input_ids=ids[:, start:start + chunk],
                              past_key_values=cache, use_cache=True)
            cache = out.past_key_values
            del out
    torch.cuda.synchronize()
    kv = torch.cuda.memory_allocated() - base
    spike = torch.cuda.max_memory_allocated() - base - kv
    del cache, ids
    return kv, spike

for chunk in [8192, 2048, 512]:
    kv, spike = prefill(chunk)
    tokens_at_once = batch * chunk
    print(f"chunk {chunk:>5}: KV cache {kv / MiB:7.1f} MiB | spike {spike / MiB:7.1f} MiB | "
          f"{spike / tokens_at_once / 1024:5.1f} KB per token in flight")

chunk  8192: KV cache  1536.0 MiB | spike  4546.1 MiB |  35.5 KB per token in flight
chunk  2048: KV cache  1536.0 MiB | spike  1256.5 MiB |  39.3 KB per token in flight
chunk   512: KV cache  1536.0 MiB | spike   650.1 MiB |  81.3 KB per token in flight


In [7]:
import collections
import torch

# 1. Record every allocation during one chunked prefill
torch.cuda.memory._record_memory_history(max_entries=200_000)
kv, spike = prefill(512)
snap = torch.cuda.memory._snapshot()
torch.cuda.memory._dump_snapshot("chunked_prefill_512.pickle")
torch.cuda.memory._record_memory_history(enabled=None)
print(f"KV cache {kv / MiB:.1f} MiB, spike {spike / MiB:.1f} MiB")

# 2. Label each allocation with the first line outside PyTorch's internals
def where(frames):
    for f in frames:
        name = f["filename"]
        if name.startswith("<"):
            return "your notebook cell"
        if name.endswith(".py") and "/torch/" not in name:
            return f"{name.split('/')[-1]}:{f['line']} in {f['name']}"
    return "unknown"

# 3. Replay the log to find the moment of peak memory
live, total, peak, peak_live = {}, 0, 0, {}
for ev in snap["device_traces"][0]:
    if ev["action"] == "alloc":
        live[ev["addr"]] = (ev["size"], ev.get("frames", []))
        total += ev["size"]
        if total > peak:
            peak, peak_live = total, dict(live)
    elif ev["action"] == "free_requested" and ev["addr"] in live:
        total -= live.pop(ev["addr"])[0]

# 4. Group what was alive at the peak by where it was created
groups = collections.defaultdict(lambda: [0, 0])
for size, frames in peak_live.values():
    g = groups[where(frames)]
    g[0] += size
    g[1] += 1

print(f"\nAlive at the peak: {peak / MiB:.1f} MiB")
for place, (size, n) in sorted(groups.items(), key=lambda x: -x[1][0])[:12]:
    print(f"  {size / MiB:8.1f} MiB in {n:4d} blocks  created at {place}")

KV cache 1536.0 MiB, spike 650.1 MiB

Alive at the peak: 2187.1 MiB
     768.0 MiB in   24 blocks  created at cache_utils.py:135 in update
     768.0 MiB in   24 blocks  created at cache_utils.py:136 in update
     448.0 MiB in    2 blocks  created at sdpa_attention.py:27 in repeat_kv
     142.0 MiB in    4 blocks  created at sdpa_attention.py:174 in sdpa_attention_forward
      14.0 MiB in    1 blocks  created at modeling_qwen2.py:348 in forward
      14.0 MiB in    1 blocks  created at modeling_qwen2.py:289 in forward
      14.0 MiB in    1 blocks  created at modeling_qwen2.py:244 in forward
      14.0 MiB in    1 blocks  created at modeling_qwen2.py:125 in apply_rotary_pos_emb
       4.0 MiB in    1 blocks  created at masking_utils.py:76 in causal_mask_function
       1.0 MiB in    1 blocks  created at 1412451495.py:16 in prefill
       0.1 MiB in    2 blocks  created at modeling_qwen2.py:94 in forward
       0.0 MiB in    1 blocks  created at modeling_qwen2.py:355 in forward


In [8]:
%cd /content/ml-memory-notes/p1-memory-breakdown
!mv chunked_prefill_512.pickle results/ 2>/dev/null; mv load_snapshot.pickle results/ 2>/dev/null
!git add results/
!git commit -m "Project 1: memory snapshots (weight load, chunked prefill); repeat_kv found at peak"
!git push

/content/ml-memory-notes/p1-memory-breakdown
[main 3aeafe3] Project 1: memory snapshots (weight load, chunked prefill); repeat_kv found at peak
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 p1-memory-breakdown/results/chunked_prefill_512.pickle
 create mode 100644 p1-memory-breakdown/results/load_snapshot.pickle
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 752.86 KiB | 3.90 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Kareem2266/ml-memory-notes.git
   7cfbbf0..3aeafe3  main -> main


In [2]:
!ls /content

sample_data


In [9]:
import time, statistics, torch
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B"
if "model" not in globals():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
cfg, dev = model.config, "cuda"

weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Memory floor for one decode step: {weight_bytes / 320e9 * 1000:.1f} ms "
      f"(reading {weight_bytes / 1e9:.2f} GB at the T4's ~320 GB/s)\n")

def timed(fn, repeats=5):
    fn()                                  # warm-up
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()          # wait for the GPU before starting the clock
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()          # wait for the GPU to finish before stopping it
        times.append(time.perf_counter() - t0)
    return statistics.median(times)

# --- Prefill: the whole prompt in one pass ----------------------------------
print("PREFILL")
for batch, seq in [(1, 2048), (1, 8192), (4, 2048)]:
    ids = torch.randint(0, cfg.vocab_size, (batch, seq), device=dev)
    def run():
        with torch.inference_mode():
            model.model(input_ids=ids, use_cache=True)
    t = timed(run)
    print(f"  {batch} x {seq:>5} tokens: {t * 1000:7.1f} ms  -> {batch * seq / t:8.0f} tokens/s")
    del ids

# --- Decode: one new token per conversation per step -------------------------
STEPS = 64
def decode_step_time(batch):
    prompt = torch.randint(0, cfg.vocab_size, (batch, 512), device=dev)
    with torch.inference_mode():
        cache = model.model(input_ids=prompt, use_cache=True).past_key_values
        tok = torch.randint(0, cfg.vocab_size, (batch, 1), device=dev)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(STEPS):
            out = model(input_ids=tok, past_key_values=cache, use_cache=True)
            cache = out.past_key_values
            tok = out.logits[:, -1:].argmax(-1)    # pick the next token
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / STEPS

print("\nDECODE (512-token context, 64 new tokens)")
for batch in [1, 4, 16]:
    decode_step_time(batch)                    # warm-up
    t = statistics.median(decode_step_time(batch) for _ in range(3))
    print(f"  {batch:>2} conversation(s): {t * 1000:6.2f} ms per step -> {batch / t:7.0f} tokens/s")

Memory floor for one decode step: 3.1 ms (reading 0.99 GB at the T4's ~320 GB/s)

PREFILL
  1 x  2048 tokens:   135.4 ms  ->    15123 tokens/s
  1 x  8192 tokens:   716.3 ms  ->    11437 tokens/s
  4 x  2048 tokens:   540.7 ms  ->    15152 tokens/s

DECODE (512-token context, 64 new tokens)
   1 conversation(s):  22.40 ms per step ->      45 tokens/s
   4 conversation(s):  28.34 ms per step ->     141 tokens/s
  16 conversation(s):  22.94 ms per step ->     697 tokens/s
